# ChurnGuard ML v2: UCI Online Retail II

This notebook trains the first machine-learning churn model for ChurnGuard using UCI Online Retail II. It produces investor-demo evidence and exports CSV files that can be imported into the Rails approval queue.

## 1. Environment setup

Run this cell once if your Jupyter kernel does not already have the ML dependencies.

In [ ]:
import sys
import subprocess

required = ["scikit-learn", "joblib", "openpyxl"]
subprocess.check_call([sys.executable, "-m", "pip", "install", *required])

In [ ]:
from pathlib import Path
import sys
import pandas as pd

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parents[1]
sys.path.insert(0, str(PROJECT_ROOT / "research" / "ml"))

from churn_pipeline import (
    FEATURE_COLUMNS,
    aggregate_orders,
    audit_raw_data,
    build_current_features,
    build_training_snapshots,
    clean_order_lines,
    export_app_files,
    load_online_retail_workbook,
    save_model_artifacts,
    score_current_customers,
    train_models,
)

DATASET_PATH = Path("/Users/serhii.kravchenko/Downloads/online_retail_II.xlsx")
EXPORT_DIR = PROJECT_ROOT / "research" / "exports"
MODEL_DIR = PROJECT_ROOT / "research" / "models"
DATASET_PATH.exists(), DATASET_PATH

## 2. Load workbook and audit source data

In [ ]:
raw = load_online_retail_workbook(DATASET_PATH)
audit = audit_raw_data(raw)
audit

## 3. Clean transactions

Rules: remove rows without customers, treat `C` invoices and negative quantities as cancellations, keep positive paid order lines, and calculate line revenue as quantity times price.

In [ ]:
paid_lines, cancellations = clean_order_lines(raw)
orders = aggregate_orders(paid_lines)

{
    "paid_line_rows": len(paid_lines),
    "paid_orders": orders["shopify_order_id"].nunique(),
    "customers": orders["shopify_customer_id"].nunique(),
    "revenue": round(float(orders["amount"].sum()), 2),
    "date_min": str(orders["ordered_at"].min()),
    "date_max": str(orders["ordered_at"].max()),
}

## 4. Build churn labels and features

A customer is labelled churned when they have no next paid order within 90 days after the snapshot date. Every feature uses only past behavior.

In [ ]:
snapshots = build_training_snapshots(orders, cancellations)
snapshots[["snapshot_date", "shopify_customer_id", "churned", *FEATURE_COLUMNS]].head()

In [ ]:
snapshots.groupby("snapshot_date").agg(
    customers=("shopify_customer_id", "nunique"),
    churn_rate=("churned", "mean"),
    revenue=("lifetime_value", "sum"),
).tail()

## 5. Train baseline and ML models

The split is time-based, not random, so the model is evaluated on later customer behavior.

In [ ]:
training_result = train_models(snapshots)
training_result.metrics

## 6. Score current customers

In [ ]:
current_features = build_current_features(orders, cancellations)
scored = score_current_customers(
    training_result.model,
    current_features,
    training_result.risk_thresholds,
)

scored.groupby("risk_tier").agg(
    customers=("shopify_customer_id", "count"),
    revenue_at_risk=("revenue_at_risk", "sum"),
    avg_score=("risk_score", "mean"),
).sort_index()

In [ ]:
scored.sort_values("risk_score", ascending=False)[[
    "shopify_customer_id",
    "risk_score",
    "risk_tier",
    "lifetime_value",
    "revenue_at_risk",
    "proposed_discount",
    "reason",
]].head(20)

## 7. Business interpretation

Use these points in the investor story:

- The model is trained on real commerce transaction history, not synthetic CRM rows.
- Churn is predicted before the next purchase fails to happen.
- Risk scores become concrete approval actions in ChurnGuard.
- The shop owner sees revenue at risk and one proposed retention action, not an analytics dashboard.

In [ ]:
business_summary = {
    "customers_scored": int(scored["shopify_customer_id"].nunique()),
    "high_risk_customers": int(scored[scored["risk_tier"] == "high"].shape[0]),
    "medium_risk_customers": int(scored[scored["risk_tier"] == "medium"].shape[0]),
    "approval_queue_revenue_at_risk": round(float(scored[scored["risk_tier"].isin(["high", "medium"])] ["revenue_at_risk"].sum()), 2),
    "model_version": scored["model_version"].iloc[0],
}
business_summary

## 8. Export app-ready files and model artifacts

In [ ]:
csv_outputs = export_app_files(orders, scored, EXPORT_DIR)
model_outputs = save_model_artifacts(training_result, MODEL_DIR)
{**csv_outputs, **model_outputs}

## 9. Upload into ChurnGuard Rails

From the project root, run:

```bash
bin/rails runner script/import_online_retail_ml_demo.rb
```

Then start the app and open the approval queue. The queue will show ML-generated high and medium risk actions using the existing ChurnGuard approval cards.